# ML-Based Stop Loss: Historical Volatility and Drawdown Recovery

**Reference:** Hands-On AI Trading with Python, QuantConnect, and AWS (2025)
Chapter 06 - Applied Machine Learning, Example 08 (p185)

## Objectifs

1. Calculer la volatilité historique rolling (30/60/90 jours) comme features
2. Construire un modèle LASSO pour prédire le rendement bas hebdomadaire
3. Comparer stop-loss fixe vs stop-loss ML-adaptatif
4. Analyser la logique de drawdown recovery et son impact sur le risque

> **[REFERENCE QC Cloud]**
> Ce notebook utilise QuantBook et nécessite l'environnement QuantConnect Cloud.
> Pour exécuter : https://www.quantconnect.com/research

In [1]:
from AlgorithmImports import *
from sklearn.linear_model import Lasso, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
import numpy as np
import pandas as pd

## 1. Chargement des données

On charge 5 ans de données quotidiennes pour KO (Coca-Cola) et SPY (proxy VIX).
KO est l'actif tradé, SPY fournit la volatilité de marché comme feature.

In [2]:
qb = QuantBook()

ko = qb.add_equity("KO", data_normalization_mode=DataNormalizationMode.RAW)
spy = qb.add_equity("SPY", data_normalization_mode=DataNormalizationMode.RAW)

# 5 years of daily data
end_date = datetime(2024, 12, 31)
start_date = end_date - timedelta(5 * 365)

history = qb.history([ko.symbol, spy.symbol], start_date, end_date, Resolution.DAILY)
df = history['close'].unstack(0)
df.columns = ['KO', 'SPY']
df.dropna(inplace=True)

print(f"Data range: {df.index[0].date()} to {df.index[-1].date()}")
print(f"Total trading days: {len(df)}")
df.head()

## 2. Feature Engineering: Volatilité Historique

On calcule 3 indicateurs de volatilité rolling :
- **Rendements SPY** (proxy VIX) : écart-type rolling sur 22 jours
- **ATR (Average True Range)** : mesure de volatilité intraday
- **Écart-type des rendements KO** : dispersion des rendements sur la période

Ces features capturent différents aspects de la volatilité.

In [3]:
# Daily returns
ko_returns = df['KO'].pct_change()
spy_returns = df['SPY'].pct_change()

# Rolling volatility at multiple lookback windows
vol_30 = ko_returns.rolling(30).std() * np.sqrt(252)   # Annualized
vol_60 = ko_returns.rolling(60).std() * np.sqrt(252)
vol_90 = ko_returns.rolling(90).std() * np.sqrt(252)

# SPY realized volatility (proxy for market-wide implied vol)
spy_vol = spy_returns.rolling(22).std() * np.sqrt(252)

# True Range (simplified daily approximation)
df_high = history['high'].unstack(0) if 'high' in history.columns else None
df_low = history['low'].unstack(0) if 'low' in history.columns else None

# Use close-to-close as proxy for ATR
ko_std = ko_returns.rolling(22).std()

# Build feature DataFrame
features = pd.DataFrame({
    'spy_vol': spy_vol,
    'ko_std': ko_std,
    'vol_30': vol_30,
    'vol_60': vol_60,
    'vol_90': vol_90,
}).dropna()

print("Feature summary statistics:")
print(features.describe().round(4))

## 3. Label Construction: Rendement Bas Hebdomadaire

Le label prédit est le rendement entre le prix d'ouverture du lundi et le prix le plus bas
atteint pendant la semaine (5 jours de trading). Ce label mesure le pire scénario hebdomadaire.

In [4]:
# Compute weekly low return: (lowest low of week - Monday open) / Monday open
weekly_groups = df['KO'].resample('W')
weekly_labels = []

for name, group in df['KO'].resample('W-FRI'):
    if len(group) < 3:
        continue
    week_open = group.iloc[0]
    week_low = group.min()
    low_return = (week_low - week_open) / week_open
    weekly_labels.append({
        'week_end': name,
        'week_open': week_open,
        'week_low': week_low,
        'low_return': low_return,
    })

labels_df = pd.DataFrame(weekly_labels).set_index('week_end')
print(f"Weekly labels computed: {len(labels_df)} weeks")
print(f"\nLow return statistics:")
print(f"  Mean:   {labels_df['low_return'].mean():.4f}")
print(f"  Median: {labels_df['low_return'].median():.4f}")
print(f"  Min:    {labels_df['low_return'].min():.4f}")
print(f"  Max:    {labels_df['low_return'].max():.4f}")
labels_df['low_return'].hist(bins=50, figsize=(10, 4))

## 4. Modèle LASSO: Prédiction du Stop-Loss Dynamique

Le modèle LASSO (Least Absolute Shrinkage and Selection Operator) est choisi car :
- Il sélectionne automatiquement les features les plus pertinentes
- Il réduit le bruit en excluant les variables moins essentielles
- La régularisation L1 prévient le surapprentissage

On utilise `LassoCV` avec validation croisée temporelle pour sélectionner le paramètre alpha optimal.

In [5]:
# Align features with weekly labels
# Use the last trading day's features before each week starts
feature_weekly = features.resample('W-FRI').last()

# Merge features and labels
merged = feature_weekly.join(labels_df['low_return'], how='inner').dropna()

feature_cols = ['spy_vol', 'ko_std', 'vol_30', 'vol_60', 'vol_90']
X = merged[feature_cols].values[:-1]  # Features from previous week
y = merged['low_return'].values[1:]   # Label for next week

# Time-series split for validation
tscv = TimeSeriesSplit(n_splits=5)

# LASSO with cross-validated alpha
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = LassoCV(cv=tscv, random_state=42, max_iter=10000)
model.fit(X_scaled, y)

print(f"Optimal alpha: {model.alpha_:.6f}")
print(f"\nFeature coefficients:")
for name, coef in zip(feature_cols, model.coef_):
    print(f"  {name:10s}: {coef:+.6f}")
print(f"\nR-squared (train): {model.score(X_scaled, y):.4f}")

## 5. Comparaison: Stop-Loss Fixe vs ML-Adaptatif

On compare deux approches :
- **Benchmark** : Stop-loss fixe à 5% sous le prix d'achat
- **ML** : Stop-loss placé sous le prix bas prédit par LASSO

La comparaison se fait sur le rendement cumulé et le drawdown maximum.

In [6]:
# Simulate both strategies
predictions = model.predict(X_scaled)

# Strategy 1: Fixed 5% stop-loss
fixed_sl_pct = 0.05
fixed_stopped = merged['low_return'].values[1:] < -fixed_sl_pct

# Strategy 2: ML-predicted stop-loss
ml_stopped = merged['low_return'].values[1:] < predictions

# Count stop-loss triggers
total_weeks = len(merged) - 1
print(f"Total weeks: {total_weeks}")
print(f"\nFixed 5% stop-loss triggered: {fixed_stopped.sum()} ({fixed_stopped.mean()*100:.1f}%)")
print(f"ML stop-loss triggered:       {ml_stopped.sum()} ({ml_stopped.mean()*100:.1f}%)")

# Simulate weekly returns with stop-loss
actual_returns = merged['low_return'].values[1:]

# When stop-loss triggers, return = -stop_distance; otherwise, actual weekly return
weekly_return_full = df['KO'].resample('W-FRI').apply(lambda x: (x.iloc[-1] - x.iloc[0]) / x.iloc[0])
weekly_returns = weekly_return_full.values[1:len(actual_returns)+1]

# Fixed stop-loss: cap loss at -5%
fixed_returns = np.where(actual_returns < -fixed_sl_pct, -fixed_sl_pct, weekly_returns[:len(actual_returns)])

# ML stop-loss: cap loss at predicted level
ml_returns = np.where(actual_returns < predictions, predictions, weekly_returns[:len(actual_returns)])

# Buy-and-hold for comparison
hold_returns = weekly_returns[:len(actual_returns)]

print(f"\nCumulative performance comparison:")
print(f"  Buy & Hold:   {(1 + hold_returns).prod() - 1:+.2%}")
print(f"  Fixed 5% SL:  {(1 + fixed_returns).prod() - 1:+.2%}")
print(f"  ML Stop-Loss: {(1 + ml_returns).prod() - 1:+.2%}")

## 6. Analyse du Drawdown Recovery

Le concept de drawdown recovery : après un stop-loss déclenché, on attend que le prix
remonte d'un certain pourcentage avant de reprendre position. Cela évite de rentrer pendant
une chute continue.

On analyse l'effet d'une politique de re-entry différée.

In [7]:
# Drawdown recovery analysis
# Compute rolling drawdown from peak
ko_prices = df['KO']
rolling_max = ko_prices.expanding().max()
drawdown = (ko_prices - rolling_max) / rolling_max

# Recovery rate: how quickly price recovers after drawdown events
drawdown_threshold = -0.05  # 5% drawdown triggers tracking
recovery_threshold = 0.02   # 2% recovery to re-enter

drawdown_events = []
in_drawdown = False
dd_start = None
dd_min_price = None

for date, dd_val in drawdown.items():
    if dd_val < drawdown_threshold and not in_drawdown:
        in_drawdown = True
        dd_start = date
        dd_min_price = ko_prices[date]
    elif in_drawdown:
        if ko_prices[date] < dd_min_price:
            dd_min_price = ko_prices[date]
        recovery_from_min = (ko_prices[date] - dd_min_price) / dd_min_price
        if recovery_from_min >= recovery_threshold:
            drawdown_events.append({
                'start': dd_start,
                'end': date,
                'duration_days': (date - dd_start).days,
                'max_drawdown': drawdown[dd_start:date].min(),
            })
            in_drawdown = False

events_df = pd.DataFrame(drawdown_events)
print(f"Drawdown events (>{abs(drawdown_threshold)*100:.0f}% decline): {len(events_df)}")
if len(events_df) > 0:
    print(f"\nRecovery statistics:")
    print(f"  Avg recovery time: {events_df['duration_days'].mean():.0f} days")
    print(f"  Avg max drawdown:  {events_df['max_drawdown'].mean():.2%}")
    print(f"  Worst drawdown:    {events_df['max_drawdown'].min():.2%}")
    print(f"\nEvents detail:")
    display(events_df)

## 7. Sensibilité du Stop-Loss

On teste différents niveaux de stop-loss fixe pour trouver le point optimal.
Le livre Broad montre que 17/20 distances testées surperforment le buy-and-hold.

In [8]:
# Sensitivity analysis: test different fixed stop-loss levels
sl_levels = np.arange(0.02, 0.15, 0.01)  # 2% to 14%
results = []

for sl_pct in sl_levels:
    stopped = actual_returns < -sl_pct
    capped_returns = np.where(stopped, -sl_pct, weekly_returns[:len(actual_returns)])
    
    cumulative = (1 + capped_returns).prod() - 1
    max_dd = np.min(np.cumsum(capped_returns))
    
    results.append({
        'stop_loss_pct': f"{sl_pct:.0%}",
        'cumulative_return': cumulative,
        'max_drawdown': max_dd,
        'triggers': stopped.sum(),
        'trigger_rate': f"{stopped.mean()*100:.1f}%",
    })

sensitivity_df = pd.DataFrame(results)
print("Stop-Loss Sensitivity Analysis:")
print(sensitivity_df.to_string(index=False))

## 8. Conclusion

| Approche | Avantage | Limite |
|----------|----------|--------|
| Stop-loss fixe (5%) | Simple, prévisible | Ne s'adapte pas à la volatilité courante |
| LASSO dynamique | Adapte le stop à la vol prédite | Peut sous-estimer les queues de distribution |
| Drawdown recovery | Évite les re-entry prématurées | Augmente le temps hors-marché |

**Points clés du livre Broad (Ch6 Ex08)** :
- Le LASSO sélectionne automatiquement les features de volatilité pertinentes
- 17/20 niveaux de stop-loss fixe surperforment le buy-and-hold
- Le Sharpe ratio s'effondre quand le stop-loss est trop serré (>98.5%)
- La version put-option (Algo 3) offre un plancher garanti mais plus coûteux

**Stratégie complète** : voir `main.py` pour l'implémentation algorithmique QC Cloud.